In [ ]:
med_raw = session.table(f"{DB}.{RAW}.STREAM_T_MEDICATIONS").filter(
    col("METADATA$ACTION") == "INSERT"
).drop("METADATA$ACTION", "METADATA$ISUPDATE", "METADATA$ROW_ID").cache_result()
print("Stream read complete")

In [ ]:
ind_cols = ["ROGER_DECISION_DRUG_IND"]
code_cols = ["MEDICATION_CLASSIFICATION_CODE"]
desc_cols = ["COMMON_NAME", "MEDICATION_DESC"]
user_cols = ["ADD_USER", "MOD_USER"]
trim_cols = ["SOURCE_FILE_NAME"]

all_cols = [c.name for c in med_raw.schema.fields]
select_exprs = []
for c in all_cols:
    if c == "LOAD_TIMESTAMP":
        select_exprs.append(col(c).alias("RAW_LOAD_TIMESTAMP"))
    elif c in ind_cols:
        select_exprs.append(coalesce(when(trim(col(c)) == lit(""), lit("N")).otherwise(trim(col(c))), lit("N")).alias(c))
    elif c in code_cols:
        select_exprs.append(upper(trim(col(c))).alias(c))
    elif c in desc_cols:
        select_exprs.append(coalesce(when(trim(col(c)) == lit(""), lit("NA")).otherwise(trim(col(c))), lit("NA")).alias(c))
    elif c in user_cols:
        select_exprs.append(coalesce(when(trim(col(c)) == lit(""), lit("SYSTEM")).otherwise(trim(col(c))), lit("SYSTEM")).alias(c))
    elif c in trim_cols:
        select_exprs.append(trim(col(c)).alias(c))
    else:
        select_exprs.append(col(c))

med_clean = med_raw.select(select_exprs)
print("Transformations applied")

In [ ]:
valid_med = med_clean.filter(col("MED_ID").is_not_null())
invalid_med = med_clean.filter(col("MED_ID").is_null()).with_column("ERROR_REASON", lit("MED_ID_NULL"))
print("Valid/invalid split defined")

In [ ]:
checksum_cols = [
    ("MED_ID", "number"), ("COMMON_NAME", "string"), ("MEDICATION_DESC", "string"),
    ("ROGER_DECISION_DRUG_IND", "string"), ("MEDICATION_CLASSIFICATION_CODE", "string"),
    ("ALLOW_FREQUENCY_NUM", "number"), ("ADD_PERSON_ID", "number"),
    ("ADD_ORGN_ID", "number"), ("MOD_PERSON_ID", "number"),
    ("MOD_ORGN_ID", "number"), ("ADD_USER", "string"), ("MOD_USER", "string")
]
checksum_exprs = []
for col_name, col_type in checksum_cols:
    if col_type == "string":
        checksum_exprs.append(coalesce(col(col_name), lit("")))
    else:
        checksum_exprs.append(coalesce(col(col_name).cast("string"), lit("")))

med_final = valid_med.with_column(
    "CHECKSUM", sha2(concat_ws(lit("|"), *checksum_exprs), 256)
).with_column("STAGING_LOADED_TIMESTAMP", current_timestamp())

med_final.write.save_as_table(f"{DB}.{STG}.{STG_MEDICATIONS}", mode="truncate")
print(f"MEDICATIONS loaded into {STG}.{STG_MEDICATIONS}")

In [ ]:
invalid_count = invalid_med.count()
if invalid_count > 0:
    invalid_med.select(
        lit("T_MEDICATIONS").alias("TABLE_NAME"), col("ERROR_REASON"),
        col("SOURCE_FILE_NAME"), col("RAW_LOAD_TIMESTAMP")
    ).write.save_as_table(f"{DB}.{STG}.{INVALID_RECORDS}", mode="append")
    print(f"Invalid records saved: {invalid_count}")
else:
    print("No invalid records")

In [ ]:
rows_processed = session.table(f"{DB}.{STG}.{STG_MEDICATIONS}").count()
status = 'SUCCESS' if invalid_count == 0 else 'PARTIAL_SUCCESS'

session.call(f"{DB}.{AUDIT}.{SP_LOG_AUDIT}",
    str(uuid.uuid4()), 'STG_MEDICATIONS_LOAD', 'STAGING',
    datetime.now(), datetime.now(), rows_processed, invalid_count, status, None)
session.call(f"{DB}.{UTIL}.{SP_SEND_PIPELINE_NOTIFICATION}",
    status, 'STG_MEDICATIONS_LOAD', 'STAGING',
    f'MEDICATIONS staging completed. Rows: {rows_processed}, Failed: {invalid_count}')
print("Audit log inserted and email sent")